Aplicação real - Motor de Risco Antifraude

In [1]:
batch_de_transacoes = [
    "PIX, 1500.50, aprovado, 14:30",
    "TED ; 200.00 ; erro ; 10:15",
    {"tipo": "DOC", "valor": 8500.00, "status": "aprovado", "horario": "11:00"},
    "PIX, -50.00, aprovado, 08:00",
    {"tipo": "PIX", "valor": 1500.00, "status": "aprovado", "horario": "02:30"},
    "  TED,   55000.00  , aprovado  , 15:00",
    {"tipo": "BOLETO", "valor": 150.00, "status": "erro", "horario": "19:00"},
    "PIX ; 25000.00 ; aprovado ; 23:15"
]


# 1 - limpaza dos dados -------------------------------
def limpeza_dado(dados):
  vendas = []
  aux = {}
  for dado in dados:
    if isinstance(dado, dict):
      dado["tipo"] = dado.get("tipo").upper()
      dado["valor"] = float(dado.get("valor"))
      dado["status"] = 'ok' if dado.get("status").strip().lower() == 'aprovado' else 'erro'
      dado["horario"] = int(dado.get("horario").split(":")[0])
      # print(dado)
      vendas.append(dado)

    elif isinstance(dado, str):
      aux = dado.replace(',',';').split(';')
      vendas.append({'tipo': aux[0].strip().upper(),
                    'valor' : float(aux[1]),
                    'status' : 'ok' if aux[2].strip().lower() == 'aprovado' else 'erro',
                    'horario': int(aux[3].split(':')[0])})

  return vendas


dados_limpos = limpeza_dado(batch_de_transacoes)

print('\ndados limpos')

print(dados_limpos)

# 2 - Lista rápida de dados com status = ok -----------

dados_ok = [dado for dado in dados_limpos if dado.get('status') == 'ok']

print('\ndados com status ok')

print(dados_ok)

# 3 - Função de avaliar risco -------------------------

def avalia_risco(valor : float, tipo : str, hora : int):
  if hora > 6 and hora < 21:
    if tipo == 'PIX':
      return 'ALTO' if valor > 1000 else 'BAIXO'
    else:
      return'BAIXO'

    return 'CRÍTICO' if valor > 20000 else 'BAIXO'

  else:
    if tipo == 'PIX':
      return 'ALTO' if valor > 5000 else 'BAIXO'

    elif tipo == 'TED':
      return 'ALTO' if valor > 10000 else 'BAIXO'

    return 'CRÍTICO' if valor > 50000 else 'BAIXO'

# testes
print('\ntestes:')
for dado in dados_ok:
  risco = avalia_risco(dado.get('valor'), dado.get('tipo'), dado.get('horario'))
  if risco == 'ALTO' or risco == 'CRÍTICO':
    print(dado)



dados limpos
[{'tipo': 'PIX', 'valor': 1500.5, 'status': 'ok', 'horario': 14}, {'tipo': 'TED', 'valor': 200.0, 'status': 'erro', 'horario': 10}, {'tipo': 'DOC', 'valor': 8500.0, 'status': 'ok', 'horario': 11}, {'tipo': 'PIX', 'valor': -50.0, 'status': 'ok', 'horario': 8}, {'tipo': 'PIX', 'valor': 1500.0, 'status': 'ok', 'horario': 2}, {'tipo': 'TED', 'valor': 55000.0, 'status': 'ok', 'horario': 15}, {'tipo': 'BOLETO', 'valor': 150.0, 'status': 'erro', 'horario': 19}, {'tipo': 'PIX', 'valor': 25000.0, 'status': 'ok', 'horario': 23}]

dados com status ok
[{'tipo': 'PIX', 'valor': 1500.5, 'status': 'ok', 'horario': 14}, {'tipo': 'DOC', 'valor': 8500.0, 'status': 'ok', 'horario': 11}, {'tipo': 'PIX', 'valor': -50.0, 'status': 'ok', 'horario': 8}, {'tipo': 'PIX', 'valor': 1500.0, 'status': 'ok', 'horario': 2}, {'tipo': 'TED', 'valor': 55000.0, 'status': 'ok', 'horario': 15}, {'tipo': 'PIX', 'valor': 25000.0, 'status': 'ok', 'horario': 23}]

testes:
{'tipo': 'PIX', 'valor': 1500.5, 'status'